In [1]:
pip install pyodbc


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Table Joueurs

In [2]:
import pandas as pd
import pyodbc
from datetime import date, datetime


In [9]:
# === Connexion SQL Server ===
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"  
    "DATABASE=Staging_ClubAfricain;"
    "UID=moeness;"       
    "PWD=azerty"  
)
cursor = conn.cursor()

In [10]:
# === Chargement CSV source ===
df_source = pd.read_csv("../Data/joueurs.csv", parse_dates=["Date_Naissance", "Date_Adhésion"])
for col in ["Taille", "Poids", "IMC", "Aspect_Général"]:
    df_source[col] = df_source[col].astype(str).str.replace(',', '.').astype(float)


In [11]:
# === Lecture table SQL existante ===
query = "SELECT * FROM Joueurs WHERE Actif = 1"
df_db = pd.read_sql(query, conn)

C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_24152\2602990041.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_db = pd.read_sql(query, conn)


In [12]:
# === Comparaison ligne par ligne ===
cols = ["Nom", "Prénom", "Date_Naissance", "Num_Licence", "Date_Adhésion",
        "Pied_Dominant", "Position", "Taille", "Poids", "IMC", "Catégorie", "Aspect_Général"]
today = date.today()

for _, new_row in df_source.iterrows():
    old = df_db[df_db["ID_Joueur"] == new_row["ID_Joueur"]]
    
    if old.empty:
        #  Nouveau joueur
        cursor.execute("""
            INSERT INTO Joueurs (
                ID_Joueur, Nom, Prénom, Date_Naissance, Num_Licence, Date_Adhésion,
                Pied_Dominant, Position, Taille, Poids, IMC, Catégorie, Aspect_Général,
                Date_Debut, Actif, Type_Changement
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, 1, 'INSERT')
        """, *[new_row[col] for col in ["ID_Joueur"] + cols], today)
        
    else:
        old_row = old.iloc[0]
        if any(str(new_row[col]) != str(old_row[col]) for col in cols):
            #  Changement détecté → historisation
            cursor.execute("""
                UPDATE Joueurs
                SET Date_Fin = ?, Actif = 0, Type_Changement = 'UPDATE', Last_Modified = ?
                WHERE ID_Joueur = ? AND Actif = 1
            """, today, datetime.now(), new_row["ID_Joueur"])

            cursor.execute("""
                INSERT INTO Joueurs (
                    ID_Joueur, Nom, Prénom, Date_Naissance, Num_Licence, Date_Adhésion,
                    Pied_Dominant, Position, Taille, Poids, IMC, Catégorie, Aspect_Général,
                    Date_Debut, Actif, Type_Changement
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, 1, 'UPDATE')
            """, *[new_row[col] for col in ["ID_Joueur"] + cols], today)

conn.commit()
cursor.close()
conn.close()

# Table Matchs

In [15]:
# === Connexion à SQL Server ===
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;" 
    "DATABASE=Staging_ClubAfricain;"
    "UID=moeness;"      
    "PWD=azerty"  
)
cursor = conn.cursor()





In [17]:
# === Charger les données Matchs depuis CSV ===
df_source = pd.read_csv("../Data/matchs.csv", parse_dates=["Date_Match"])

# === Récupérer les données actives de la table Matchs ===
df_db = pd.read_sql("SELECT * FROM Matchs WHERE Actif = 1", conn)

# === Colonnes à comparer ===
cols = [
    "ID_Match", "ID_Joueur", "Saison", "Date_Match", "Catégorie", "Compétition",
    "Titulaire", "Temps_Jeu", "Carton_Jaune", "Carton_Rouge",
    "Fautes_Commisses", "Buts", "Score", "Résultat"
]

today = date.today()

C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_24152\3576435167.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_db = pd.read_sql("SELECT * FROM Matchs WHERE Actif = 1", conn)


In [18]:
for _, new_row in df_source.iterrows():
    old = df_db[df_db["ID_Match_Unique"] == new_row["ID_Match_Unique"]]

    if old.empty:
        #  Nouveau match
        cursor.execute("""
            INSERT INTO Matchs (
                ID_Match_Unique, ID_Match, ID_Joueur, Saison, Date_Match, Catégorie,
                Compétition, Titulaire, Temps_Jeu, Carton_Jaune, Carton_Rouge,
                Fautes_Commisses, Buts, Score, Résultat,
                Date_Debut, Actif, Type_Changement
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, 1, 'INSERT')
        """, *[new_row[col] for col in ["ID_Match_Unique"] + cols], today)
    else:
        old_row = old.iloc[0]
        if any(str(new_row[col]) != str(old_row[col]) for col in cols):
            #  Historisation
            cursor.execute("""
                UPDATE Matchs
                SET Date_Fin = ?, Actif = 0, Type_Changement = 'UPDATE', Last_Modified = ?
                WHERE ID_Match_Unique = ? AND Actif = 1
            """, today, datetime.now(), new_row["ID_Match_Unique"])

            cursor.execute("""
                INSERT INTO Matchs (
                    ID_Match_Unique, ID_Match, ID_Joueur, Saison, Date_Match, Catégorie,
                    Compétition, Titulaire, Temps_Jeu, Carton_Jaune, Carton_Rouge,
                    Fautes_Commisses, Buts, Score, Résultat,
                    Date_Debut, Actif, Type_Changement
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, 1, 'UPDATE')
            """, *[new_row[col] for col in ["ID_Match_Unique"] + cols], today)

conn.commit()
cursor.close()
conn.close()


# Table Données Athlétiques

In [20]:
# Connexion SQL Server
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"  
    "DATABASE=Staging_ClubAfricain;"
    "UID=moeness;"      
    "PWD=azerty"  
)
cursor = conn.cursor()


In [22]:
# Charger les données CSV
df_source = pd.read_csv("../Data/donnees_athletiques.csv")

# Normaliser les colonnes float si nécessaire
float_cols = ["VO2_Max", "Distance_Parcourue", "Vitesse_Max", "Vitesse_Moyenne",
              "Hauteur_Saut", "Puissance_Membres", "Force_Maximale", "Temps_Récupération"]
for col in float_cols:
    df_source[col] = df_source[col].astype(str).str.replace(',', '.').astype(float)

# Récupérer les données actives dans la base
df_db = pd.read_sql("SELECT * FROM Donnees_Athletiques WHERE Actif = 1", conn)

# Colonnes à comparer
cols = ["ID_Match", "ID_Joueur", "VO2_Max", "Distance_Parcourue", "Vitesse_Max",
        "Vitesse_Moyenne", "Hauteur_Saut", "Puissance_Membres", "Force_Maximale", "Temps_Récupération"]

today = date.today()

C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_24152\2524168508.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_db = pd.read_sql("SELECT * FROM Donnees_Athletiques WHERE Actif = 1", conn)


In [23]:
for _, new_row in df_source.iterrows():
    old = df_db[df_db["ID_Athletique"] == new_row["ID_Athletique"]]

    if old.empty:
        #  Nouvelle ligne
        cursor.execute("""
            INSERT INTO Donnees_Athletiques (
                ID_Athletique, ID_Match, ID_Joueur, VO2_Max, Distance_Parcourue,
                Vitesse_Max, Vitesse_Moyenne, Hauteur_Saut, Puissance_Membres,
                Force_Maximale, Temps_Récupération,
                Date_Debut, Actif, Type_Changement
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, 1, 'INSERT')
        """, *[new_row[col] for col in ["ID_Athletique"] + cols], today)
    else:
        old_row = old.iloc[0]
        if any(str(new_row[col]) != str(old_row[col]) for col in cols):
            #  Historisation
            cursor.execute("""
                UPDATE Donnees_Athletiques
                SET Date_Fin = ?, Actif = 0, Type_Changement = 'UPDATE', Last_Modified = ?
                WHERE ID_Athletique = ? AND Actif = 1
            """, today, datetime.now(), new_row["ID_Athletique"])

            cursor.execute("""
                INSERT INTO Donnees_Athletiques (
                    ID_Athletique, ID_Match, ID_Joueur, VO2_Max, Distance_Parcourue,
                    Vitesse_Max, Vitesse_Moyenne, Hauteur_Saut, Puissance_Membres,
                    Force_Maximale, Temps_Récupération,
                    Date_Debut, Actif, Type_Changement
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, 1, 'UPDATE')
            """, *[new_row[col] for col in ["ID_Athletique"] + cols], today)

conn.commit()
cursor.close()
conn.close()

# Table Données Techniques

In [1]:
import pandas as pd
import pyodbc
from datetime import date, datetime


In [2]:
# Connexion SQL Server
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"  
    "DATABASE=Staging_ClubAfricain;"
    "UID=moeness;"      
    "PWD=azerty"  
)
cursor = conn.cursor()

In [4]:
# Charger CSV Donnees_Techniques
df_source = pd.read_csv("../Data/donnees_techniques.csv")

# Conversion des floats (si nécessaires)
float_cols = ["Précision_Passes", "Contrôle_Balle", "Précision_Tirs", "Conversion_Occasions"]
for col in float_cols:
    df_source[col] = df_source[col].astype(str).str.replace(',', '.').astype(float)

# Lire les lignes actives depuis la base
df_db = pd.read_sql("SELECT * FROM Donnees_Techniques WHERE Actif = 1", conn)

# Colonnes à comparer
cols = [
    "ID_Match", "ID_Joueur", "Nombre_Passes", "Précision_Passes", "Réception_Réussie",
    "Contrôle_Balle", "Nombre_Tirs", "Précision_Tirs", "Duels_Aériens_Gagnés",
    "Centres_Réussis", "Dribbles_Réussis", "Interceptions", "Tacles_Réussis", "Conversion_Occasions"
]

today = date.today()

C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_11300\3293079575.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_db = pd.read_sql("SELECT * FROM Donnees_Techniques WHERE Actif = 1", conn)


In [5]:
# Traitement ligne par ligne
for _, new_row in df_source.iterrows():
    old = df_db[df_db["ID_Technique"] == new_row["ID_Technique"]]

    if old.empty:
        #  Nouvelle donnée technique
        cursor.execute("""
            INSERT INTO Donnees_Techniques (
                ID_Technique, ID_Match, ID_Joueur, Nombre_Passes, Précision_Passes, Réception_Réussie,
                Contrôle_Balle, Nombre_Tirs, Précision_Tirs, Duels_Aériens_Gagnés, Centres_Réussis,
                Dribbles_Réussis, Interceptions, Tacles_Réussis, Conversion_Occasions,
                Date_Debut, Actif, Type_Changement
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, 1, 'INSERT')
        """, *[new_row[col] for col in ["ID_Technique"] + cols], today)
    else:
        old_row = old.iloc[0]
        if any(str(new_row[col]) != str(old_row[col]) for col in cols):
            #  Historiser l’ancienne version
            cursor.execute("""
                UPDATE Donnees_Techniques
                SET Date_Fin = ?, Actif = 0, Type_Changement = 'UPDATE', Last_Modified = ?
                WHERE ID_Technique = ? AND Actif = 1
            """, today, datetime.now(), new_row["ID_Technique"])

            cursor.execute("""
                INSERT INTO Donnees_Techniques (
                    ID_Technique, ID_Match, ID_Joueur, Nombre_Passes, Précision_Passes, Réception_Réussie,
                    Contrôle_Balle, Nombre_Tirs, Précision_Tirs, Duels_Aériens_Gagnés, Centres_Réussis,
                    Dribbles_Réussis, Interceptions, Tacles_Réussis, Conversion_Occasions,
                    Date_Debut, Actif, Type_Changement
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, 1, 'UPDATE')
            """, *[new_row[col] for col in ["ID_Technique"] + cols], today)

conn.commit()
cursor.close()
conn.close()

# Table Données Tactiques

In [6]:
# Connexion SQL Server
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"  
    "DATABASE=Staging_ClubAfricain;"
    "UID=moeness;"      
    "PWD=azerty"  
)
cursor = conn.cursor()

In [7]:
# Charger le CSV source
df_source = pd.read_csv("../Data/donnees_tactiques.csv")

# Lire les enregistrements actifs
df_db = pd.read_sql("SELECT * FROM Donnees_Tactiques WHERE Actif = 1", conn)

# Colonnes à comparer
cols = ["ID_Match", "ID_Joueur", "Zones_Influence", "Transitions_Rapides", "Pressions_Réussies"]
today = date.today()

C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_11300\195071114.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_db = pd.read_sql("SELECT * FROM Donnees_Tactiques WHERE Actif = 1", conn)


In [8]:
for _, new_row in df_source.iterrows():
    old = df_db[df_db["ID_Tactique"] == new_row["ID_Tactique"]]

    if old.empty:
        #  Nouvelle donnée tactique
        cursor.execute("""
            INSERT INTO Donnees_Tactiques (
                ID_Tactique, ID_Match, ID_Joueur, Zones_Influence,
                Transitions_Rapides, Pressions_Réussies,
                Date_Debut, Actif, Type_Changement
            ) VALUES (?, ?, ?, ?, ?, ?, ?, 1, 'INSERT')
        """, new_row["ID_Tactique"], new_row["ID_Match"], new_row["ID_Joueur"],
             new_row["Zones_Influence"], new_row["Transitions_Rapides"],
             new_row["Pressions_Réussies"], today)
    else:
        old_row = old.iloc[0]
        if any(str(new_row[col]) != str(old_row[col]) for col in cols):
            #  Historisation
            cursor.execute("""
                UPDATE Donnees_Tactiques
                SET Date_Fin = ?, Actif = 0, Type_Changement = 'UPDATE', Last_Modified = ?
                WHERE ID_Tactique = ? AND Actif = 1
            """, today, datetime.now(), new_row["ID_Tactique"])

            cursor.execute("""
                INSERT INTO Donnees_Tactiques (
                    ID_Tactique, ID_Match, ID_Joueur, Zones_Influence,
                    Transitions_Rapides, Pressions_Réussies,
                    Date_Debut, Actif, Type_Changement
                ) VALUES (?, ?, ?, ?, ?, ?, ?, 1, 'UPDATE')
            """, new_row["ID_Tactique"], new_row["ID_Match"], new_row["ID_Joueur"],
                 new_row["Zones_Influence"], new_row["Transitions_Rapides"],
                 new_row["Pressions_Réussies"], today)

conn.commit()
cursor.close()
conn.close()

# Table Données Psychologiques

In [15]:
# Connexion SQL Server
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"  
    "DATABASE=Staging_ClubAfricain;"
    "UID=moeness;"     
    "PWD=azerty"  
)
cursor = conn.cursor()

In [16]:
# Chargement du fichier CSV
df_source = pd.read_csv("../Data/donnees_psychologiques.csv")

# Forcer les types vers Python natif
df_source = df_source.astype({
    "ID_Psychologique": int,
    "ID_Match": int,
    "ID_Joueur": int,
    "Confiance": int,
    "Résilience": int
})

# Lire les enregistrements actifs
df_db = pd.read_sql("SELECT * FROM Donnees_Psychologiques WHERE Actif = 1", conn)

# Colonnes à comparer
cols = ["ID_Match", "ID_Joueur", "Confiance", "Résilience"]
today = date.today()

C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_11300\3445062474.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_db = pd.read_sql("SELECT * FROM Donnees_Psychologiques WHERE Actif = 1", conn)


In [17]:
for _, new_row in df_source.iterrows():
    old = df_db[df_db["ID_Psychologique"] == new_row["ID_Psychologique"]]

    if old.empty:
        #  Nouvelle donnée psychologique
        cursor.execute("""
            INSERT INTO Donnees_Psychologiques (
                ID_Psychologique, ID_Match, ID_Joueur, Confiance, Résilience,
                Date_Debut, Actif, Type_Changement
            ) VALUES (?, ?, ?, ?, ?, ?, 1, 'INSERT')
        """,
        int(new_row["ID_Psychologique"]),
        int(new_row["ID_Match"]),
        int(new_row["ID_Joueur"]),
        int(new_row["Confiance"]),
        int(new_row["Résilience"]),
        today
        )
    else:
        old_row = old.iloc[0]
        if any(str(new_row[col]) != str(old_row[col]) for col in cols):
            #  Historisation
            cursor.execute("""
                UPDATE Donnees_Psychologiques
                SET Date_Fin = ?, Actif = 0, Type_Changement = 'UPDATE', Last_Modified = ?
                WHERE ID_Psychologique = ? AND Actif = 1
            """, today, datetime.now(), int(new_row["ID_Psychologique"]))

            cursor.execute("""
                INSERT INTO Donnees_Psychologiques (
                    ID_Psychologique, ID_Match, ID_Joueur, Confiance, Résilience,
                    Date_Debut, Actif, Type_Changement
                ) VALUES (?, ?, ?, ?, ?, ?, 1, 'UPDATE')
            """,
            int(new_row["ID_Psychologique"]),
            int(new_row["ID_Match"]),
            int(new_row["ID_Joueur"]),
            int(new_row["Confiance"]),
            int(new_row["Résilience"]),
            today
            )

conn.commit()
cursor.close()
conn.close()

# Table Données Contextuelles 

In [18]:
# Connexion SQL Server
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"  
    "DATABASE=Staging_ClubAfricain;"
    "UID=moeness;"     
    "PWD=azerty"  
)
cursor = conn.cursor()

In [19]:
# Charger le CSV source
df_source = pd.read_csv("../Data/donnees_contextuelles.csv")

# Forcer les types vers Python natif
df_source = df_source.astype({
    "ID_Contexte": int,
    "ID_Match": int,
    "Saison": str,
    "Météo": str,
    "Etat_Terrain": str,
    "Adversaire": str,
    "Compétition": str
})

# Lire les enregistrements actifs
df_db = pd.read_sql("SELECT * FROM Donnees_Contextuelles WHERE Actif = 1", conn)

# Colonnes à comparer
cols = ["ID_Match", "Saison", "Météo", "Etat_Terrain", "Adversaire", "Compétition"]
today = date.today()

C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_11300\1491688563.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_db = pd.read_sql("SELECT * FROM Donnees_Contextuelles WHERE Actif = 1", conn)


In [20]:
for _, new_row in df_source.iterrows():
    old = df_db[df_db["ID_Contexte"] == new_row["ID_Contexte"]]

    if old.empty:
        #  Nouvelle donnée contextuelle
        cursor.execute("""
            INSERT INTO Donnees_Contextuelles (
                ID_Contexte, ID_Match, Saison, Météo, Etat_Terrain, Adversaire, Compétition,
                Date_Debut, Actif, Type_Changement
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, 1, 'INSERT')
        """,
        int(new_row["ID_Contexte"]),
        int(new_row["ID_Match"]),
        new_row["Saison"],
        new_row["Météo"],
        new_row["Etat_Terrain"],
        new_row["Adversaire"],
        new_row["Compétition"],
        today
        )
    else:
        old_row = old.iloc[0]
        if any(str(new_row[col]) != str(old_row[col]) for col in cols):
            #  Historisation
            cursor.execute("""
                UPDATE Donnees_Contextuelles
                SET Date_Fin = ?, Actif = 0, Type_Changement = 'UPDATE', Last_Modified = ?
                WHERE ID_Contexte = ? AND Actif = 1
            """, today, datetime.now(), int(new_row["ID_Contexte"]))

            cursor.execute("""
                INSERT INTO Donnees_Contextuelles (
                    ID_Contexte, ID_Match, Saison, Météo, Etat_Terrain, Adversaire, Compétition,
                    Date_Debut, Actif, Type_Changement
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, 1, 'UPDATE')
            """,
            int(new_row["ID_Contexte"]),
            int(new_row["ID_Match"]),
            new_row["Saison"],
            new_row["Météo"],
            new_row["Etat_Terrain"],
            new_row["Adversaire"],
            new_row["Compétition"],
            today
            )

conn.commit()
cursor.close()
conn.close()

# Table Examen General

In [21]:
# Connexion SQL Server
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"  
    "DATABASE=Staging_ClubAfricain;"
    "UID=moeness;"      
    "PWD=azerty" 
)
cursor = conn.cursor()

In [22]:
# Chargement CSV
df_source = pd.read_csv("../Data/examen_general.csv", parse_dates=["Date_Examen"])

# Conversion vers types natifs
df_source = df_source.astype({
    "ID_General": int,
    "ID_Joueur": int,
    "Température": float,
    "Tension": str,
    "Glycémie": float,
    "Diabète_Type_1": bool,
    "Diabète_Type_2": bool,
    "Asthme": bool,
    "Hypertension": bool,
    "Allergies_Sévères": bool
})

# Lecture des enregistrements actifs
df_db = pd.read_sql("SELECT * FROM Examen_General WHERE Actif = 1", conn)

# Colonnes à comparer
cols = [
    "ID_Joueur", "Date_Examen", "Température", "Tension", "Glycémie",
    "Diabète_Type_1", "Diabète_Type_2", "Asthme", "Hypertension", "Allergies_Sévères"
]
today = date.today()

C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_11300\931259246.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_db = pd.read_sql("SELECT * FROM Examen_General WHERE Actif = 1", conn)


In [23]:
# Traitement ligne par ligne
for _, new_row in df_source.iterrows():
    old = df_db[df_db["ID_General"] == new_row["ID_General"]]

    if old.empty:
        #  Nouvelle ligne
        cursor.execute("""
            INSERT INTO Examen_General (
                ID_General, ID_Joueur, Date_Examen, Température, Tension, Glycémie,
                Diabète_Type_1, Diabète_Type_2, Asthme, Hypertension, Allergies_Sévères,
                Date_Debut, Actif, Type_Changement
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, 1, 'INSERT')
        """,
        int(new_row["ID_General"]),
        int(new_row["ID_Joueur"]),
        new_row["Date_Examen"],
        float(new_row["Température"]),
        new_row["Tension"],
        float(new_row["Glycémie"]),
        bool(new_row["Diabète_Type_1"]),
        bool(new_row["Diabète_Type_2"]),
        bool(new_row["Asthme"]),
        bool(new_row["Hypertension"]),
        bool(new_row["Allergies_Sévères"]),
        today
        )
    else:
        old_row = old.iloc[0]
        if any(str(new_row[col]) != str(old_row[col]) for col in cols):
            #  Historisation
            cursor.execute("""
                UPDATE Examen_General
                SET Date_Fin = ?, Actif = 0, Type_Changement = 'UPDATE', Last_Modified = ?
                WHERE ID_General = ? AND Actif = 1
            """, today, datetime.now(), int(new_row["ID_General"]))

            cursor.execute("""
                INSERT INTO Examen_General (
                    ID_General, ID_Joueur, Date_Examen, Température, Tension, Glycémie,
                    Diabète_Type_1, Diabète_Type_2, Asthme, Hypertension, Allergies_Sévères,
                    Date_Debut, Actif, Type_Changement
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, 1, 'UPDATE')
            """,
            int(new_row["ID_General"]),
            int(new_row["ID_Joueur"]),
            new_row["Date_Examen"],
            float(new_row["Température"]),
            new_row["Tension"],
            float(new_row["Glycémie"]),
            bool(new_row["Diabète_Type_1"]),
            bool(new_row["Diabète_Type_2"]),
            bool(new_row["Asthme"]),
            bool(new_row["Hypertension"]),
            bool(new_row["Allergies_Sévères"]),
            today
            )

conn.commit()
cursor.close()
conn.close()

# Table Examen Cardiopulmonaire

In [24]:
# Connexion SQL Server
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"  
    "DATABASE=Staging_ClubAfricain;"
    "UID=moeness;"      
    "PWD=azerty"  
)
cursor = conn.cursor()

In [25]:
# Charger CSV
df_source = pd.read_csv("../Data/examen_cardiopulmonaire.csv", parse_dates=["Date_Examen"])

# Conversion vers types natifs Python
df_source = df_source.astype({
    "ID_Cardio": int,
    "ID_Joueur": int,
    "Fréquence_Cardiaque": int,
    "Capacité_Pulmonaire": float,
    "ECG_Anormal": bool,
    "ETT_Anormal": bool,
    "Test_Effort": float
})

# Récupération des lignes actives
df_db = pd.read_sql("SELECT * FROM Examen_Cardiopulmonaire WHERE Actif = 1", conn)

# Colonnes à comparer
cols = [
    "ID_Joueur", "Date_Examen", "Fréquence_Cardiaque", "Capacité_Pulmonaire",
    "ECG_Anormal", "ETT_Anormal", "Test_Effort"
]

today = date.today()

C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_11300\600496082.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_db = pd.read_sql("SELECT * FROM Examen_Cardiopulmonaire WHERE Actif = 1", conn)


In [26]:
for _, new_row in df_source.iterrows():
    old = df_db[df_db["ID_Cardio"] == new_row["ID_Cardio"]]

    if old.empty:
        #  Nouvelle entrée
        cursor.execute("""
            INSERT INTO Examen_Cardiopulmonaire (
                ID_Cardio, ID_Joueur, Date_Examen, Fréquence_Cardiaque, Capacité_Pulmonaire,
                ECG_Anormal, ETT_Anormal, Test_Effort,
                Date_Debut, Actif, Type_Changement
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, 1, 'INSERT')
        """,
        int(new_row["ID_Cardio"]),
        int(new_row["ID_Joueur"]),
        new_row["Date_Examen"],
        int(new_row["Fréquence_Cardiaque"]),
        float(new_row["Capacité_Pulmonaire"]),
        bool(new_row["ECG_Anormal"]),
        bool(new_row["ETT_Anormal"]),
        float(new_row["Test_Effort"]),
        today
        )
    else:
        old_row = old.iloc[0]
        if any(str(new_row[col]) != str(old_row[col]) for col in cols):
            #  Historisation
            cursor.execute("""
                UPDATE Examen_Cardiopulmonaire
                SET Date_Fin = ?, Actif = 0, Type_Changement = 'UPDATE', Last_Modified = ?
                WHERE ID_Cardio = ? AND Actif = 1
            """, today, datetime.now(), int(new_row["ID_Cardio"]))

            cursor.execute("""
                INSERT INTO Examen_Cardiopulmonaire (
                    ID_Cardio, ID_Joueur, Date_Examen, Fréquence_Cardiaque, Capacité_Pulmonaire,
                    ECG_Anormal, ETT_Anormal, Test_Effort,
                    Date_Debut, Actif, Type_Changement
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, 1, 'UPDATE')
            """,
            int(new_row["ID_Cardio"]),
            int(new_row["ID_Joueur"]),
            new_row["Date_Examen"],
            int(new_row["Fréquence_Cardiaque"]),
            float(new_row["Capacité_Pulmonaire"]),
            bool(new_row["ECG_Anormal"]),
            bool(new_row["ETT_Anormal"]),
            float(new_row["Test_Effort"]),
            today
            )

conn.commit()
cursor.close()
conn.close()

# Table Examen Locomoteur

In [27]:
# Connexion SQL Server
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"  
    "DATABASE=Staging_ClubAfricain;"
    "UID=moeness;"      
    "PWD=azerty"  
)
cursor = conn.cursor()

In [28]:
# Charger CSV
df_source = pd.read_csv("../Data/examen_locomoteur.csv", parse_dates=["Date_Examen"])

# Conversion des types
df_source = df_source.astype({
    "ID_Locomoteur": int,
    "ID_Joueur": int,
    "Lésion_Musculaire": bool,
    "Douleurs_Articulaires": bool,
    "Tendinites_Chroniques": bool,
    "Déformation_Squelettique": bool,
    "Opération": bool
})

# Lire les enregistrements actifs
df_db = pd.read_sql("SELECT * FROM Examen_Locomoteur WHERE Actif = 1", conn)

# Colonnes à comparer
cols = [
    "ID_Joueur", "Date_Examen", "Lésion_Musculaire", "Douleurs_Articulaires",
    "Tendinites_Chroniques", "Déformation_Squelettique", "Opération"
]
today = date.today()

C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_11300\1815753397.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_db = pd.read_sql("SELECT * FROM Examen_Locomoteur WHERE Actif = 1", conn)


In [29]:
# Parcourir chaque ligne source
for _, new_row in df_source.iterrows():
    old = df_db[df_db["ID_Locomoteur"] == new_row["ID_Locomoteur"]]

    if old.empty:
        #  Nouveau résultat
        cursor.execute("""
            INSERT INTO Examen_Locomoteur (
                ID_Locomoteur, ID_Joueur, Date_Examen,
                Lésion_Musculaire, Douleurs_Articulaires,
                Tendinites_Chroniques, Déformation_Squelettique, Opération,
                Date_Debut, Actif, Type_Changement
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, 1, 'INSERT')
        """,
        int(new_row["ID_Locomoteur"]),
        int(new_row["ID_Joueur"]),
        new_row["Date_Examen"],
        bool(new_row["Lésion_Musculaire"]),
        bool(new_row["Douleurs_Articulaires"]),
        bool(new_row["Tendinites_Chroniques"]),
        bool(new_row["Déformation_Squelettique"]),
        bool(new_row["Opération"]),
        today
        )
    else:
        old_row = old.iloc[0]
        if any(str(new_row[col]) != str(old_row[col]) for col in cols):
            #  Historisation
            cursor.execute("""
                UPDATE Examen_Locomoteur
                SET Date_Fin = ?, Actif = 0, Type_Changement = 'UPDATE', Last_Modified = ?
                WHERE ID_Locomoteur = ? AND Actif = 1
            """, today, datetime.now(), int(new_row["ID_Locomoteur"]))

            cursor.execute("""
                INSERT INTO Examen_Locomoteur (
                    ID_Locomoteur, ID_Joueur, Date_Examen,
                    Lésion_Musculaire, Douleurs_Articulaires,
                    Tendinites_Chroniques, Déformation_Squelettique, Opération,
                    Date_Debut, Actif, Type_Changement
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, 1, 'UPDATE')
            """,
            int(new_row["ID_Locomoteur"]),
            int(new_row["ID_Joueur"]),
            new_row["Date_Examen"],
            bool(new_row["Lésion_Musculaire"]),
            bool(new_row["Douleurs_Articulaires"]),
            bool(new_row["Tendinites_Chroniques"]),
            bool(new_row["Déformation_Squelettique"]),
            bool(new_row["Opération"]),
            today
            )

conn.commit()
cursor.close()
conn.close()

# Table Examen ORL

In [31]:
# Connexion SQL Server
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"  
    "DATABASE=Staging_ClubAfricain;"
    "UID=moeness;"      
    "PWD=azerty"  
)
cursor = conn.cursor()

In [32]:
# Charger CSV
df_source = pd.read_csv("../Data/examen_orl.csv", parse_dates=["Date_Examen"])

# Conversion types natifs
df_source = df_source.astype({
    "ID_ORL": int,
    "ID_Joueur": int,
    "Audition": float,
    "Respiration_Nasale": int,
    "État_Gorge": int
})

# Lire les lignes actives
df_db = pd.read_sql("SELECT * FROM Examen_ORL WHERE Actif = 1", conn)

# Colonnes à comparer
cols = ["ID_Joueur", "Date_Examen", "Audition", "Respiration_Nasale", "État_Gorge"]
today = date.today()

C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_11300\3766958483.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_db = pd.read_sql("SELECT * FROM Examen_ORL WHERE Actif = 1", conn)


In [33]:
for _, new_row in df_source.iterrows():
    old = df_db[df_db["ID_ORL"] == new_row["ID_ORL"]]

    if old.empty:
        #  Nouvelle entrée
        cursor.execute("""
            INSERT INTO Examen_ORL (
                ID_ORL, ID_Joueur, Date_Examen, Audition, Respiration_Nasale, État_Gorge,
                Date_Debut, Actif, Type_Changement
            ) VALUES (?, ?, ?, ?, ?, ?, ?, 1, 'INSERT')
        """,
        int(new_row["ID_ORL"]),
        int(new_row["ID_Joueur"]),
        new_row["Date_Examen"],
        float(new_row["Audition"]),
        int(new_row["Respiration_Nasale"]),
        int(new_row["État_Gorge"]),
        today
        )
    else:
        old_row = old.iloc[0]
        if any(str(new_row[col]) != str(old_row[col]) for col in cols):
            #  Historisation
            cursor.execute("""
                UPDATE Examen_ORL
                SET Date_Fin = ?, Actif = 0, Type_Changement = 'UPDATE', Last_Modified = ?
                WHERE ID_ORL = ? AND Actif = 1
            """, today, datetime.now(), int(new_row["ID_ORL"]))

            cursor.execute("""
                INSERT INTO Examen_ORL (
                    ID_ORL, ID_Joueur, Date_Examen, Audition, Respiration_Nasale, État_Gorge,
                    Date_Debut, Actif, Type_Changement
                ) VALUES (?, ?, ?, ?, ?, ?, ?, 1, 'UPDATE')
            """,
            int(new_row["ID_ORL"]),
            int(new_row["ID_Joueur"]),
            new_row["Date_Examen"],
            float(new_row["Audition"]),
            int(new_row["Respiration_Nasale"]),
            int(new_row["État_Gorge"]),
            today
            )

conn.commit()
cursor.close()
conn.close()

# Table Examen Stomatologique

In [34]:
# Connexion SQL Server
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"  
    "DATABASE=Staging_ClubAfricain;"
    "UID=moeness;"      
    "PWD=azerty"  
)
cursor = conn.cursor()

In [35]:
# Chargement CSV
df_source = pd.read_csv("../Data/examen_stomatologique.csv", parse_dates=["Date_Examen"])

# Conversion des types
df_source = df_source.astype({
    "ID_Stomatologique": int,
    "ID_Joueur": int,
    "Nombre_Caries": int,
    "État_Gencives": int,
    "Problèmes_Mâchoire": int
})

# Lire les données actives
df_db = pd.read_sql("SELECT * FROM Examen_Stomatologique WHERE Actif = 1", conn)

# Colonnes à comparer
cols = ["ID_Joueur", "Date_Examen", "Nombre_Caries", "État_Gencives", "Problèmes_Mâchoire"]
today = date.today()

C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_11300\2485975430.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_db = pd.read_sql("SELECT * FROM Examen_Stomatologique WHERE Actif = 1", conn)


In [36]:
for _, new_row in df_source.iterrows():
    old = df_db[df_db["ID_Stomatologique"] == new_row["ID_Stomatologique"]]

    if old.empty:
        #  Nouvelle entrée
        cursor.execute("""
            INSERT INTO Examen_Stomatologique (
                ID_Stomatologique, ID_Joueur, Date_Examen,
                Nombre_Caries, État_Gencives, Problèmes_Mâchoire,
                Date_Debut, Actif, Type_Changement
            ) VALUES (?, ?, ?, ?, ?, ?, ?, 1, 'INSERT')
        """,
        int(new_row["ID_Stomatologique"]),
        int(new_row["ID_Joueur"]),
        new_row["Date_Examen"],
        int(new_row["Nombre_Caries"]),
        int(new_row["État_Gencives"]),
        int(new_row["Problèmes_Mâchoire"]),
        today
        )
    else:
        old_row = old.iloc[0]
        if any(str(new_row[col]) != str(old_row[col]) for col in cols):
            # Historisation
            cursor.execute("""
                UPDATE Examen_Stomatologique
                SET Date_Fin = ?, Actif = 0, Type_Changement = 'UPDATE', Last_Modified = ?
                WHERE ID_Stomatologique = ? AND Actif = 1
            """, today, datetime.now(), int(new_row["ID_Stomatologique"]))

            cursor.execute("""
                INSERT INTO Examen_Stomatologique (
                    ID_Stomatologique, ID_Joueur, Date_Examen,
                    Nombre_Caries, État_Gencives, Problèmes_Mâchoire,
                    Date_Debut, Actif, Type_Changement
                ) VALUES (?, ?, ?, ?, ?, ?, ?, 1, 'UPDATE')
            """,
            int(new_row["ID_Stomatologique"]),
            int(new_row["ID_Joueur"]),
            new_row["Date_Examen"],
            int(new_row["Nombre_Caries"]),
            int(new_row["État_Gencives"]),
            int(new_row["Problèmes_Mâchoire"]),
            today
            )

conn.commit()
cursor.close()
conn.close()